# Data Combiner et Chargement SQL (Schéma en Étoile)

Ce notebook fusionne les données brutes issues des scrapers Avito et Mubawab, les nettoie, et les charge dans la base de données PostgreSQL selon une architecture en **Schéma en Étoile** (Star Schema).

## 1. Importations et Configuration

In [ ]:
import os
import glob
import logging
from datetime import datetime
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

logging.basicConfig(level=logging.INFO, format="%(asctime)s \u2502 %(message)s")
logger = logging.getLogger(__name__)

# Configuration DB (Modifiez si nécessaire)
DB_CONFIG = {
    "host":     "localhost",
    "port":     5433,
    "dbname":   "immobilier_maroc",
    "user":     "immobilier",
    "password": "immobilier123"
}

ROOT = "../"
DATA_RAW_AVITO = os.path.join(ROOT, "data", "raw", "avito")
DATA_RAW_MUBAWAB = os.path.join(ROOT, "data", "raw", "mubawab")
DATA_PROCESSED = os.path.join(ROOT, "data", "processed")

## 2. Fusion et Nettoyage des Données (Data Combiner)

In [ ]:
class DataCombiner:
    FINAL_SCHEMA = {
        'id_annonce': str, 'source': str, 'type_bien': str, 'ville': str, 'prix': float,
        'surface_m2': int, 'nb_chambres': str, 'nb_salles_bain': str, 'etage': str,
        'parking': int, 'ascenseur': int, 'balcon': int, 'piscine': int, 'jardin': int,
        'titre': str, 'description': str, 'url': str, 'date_scraping': str
    }
    
    def load_latest_files(self):
        avito_files = glob.glob(os.path.join(DATA_RAW_AVITO, "avito_*.csv"))
        df_avito = pd.read_csv(max(avito_files, key=os.path.getctime)) if avito_files else pd.DataFrame()
        
        mubawab_files = glob.glob(os.path.join(DATA_RAW_MUBAWAB, "mubawab_*.csv"))
        df_mubawab = pd.read_csv(max(mubawab_files, key=os.path.getctime)) if mubawab_files else pd.DataFrame()
        
        return df_avito, df_mubawab

    def normalize(self, df):
        if df.empty: return pd.DataFrame(columns=self.FINAL_SCHEMA.keys())
        df_norm = pd.DataFrame()
        for col, dtype in self.FINAL_SCHEMA.items():
            if col in df.columns:
                df_norm[col] = df[col]
            else:
                df_norm[col] = 0 if dtype == int else (None if dtype == float else "")
        return df_norm

    def clean_data(self, df):
        df = df.drop_duplicates(subset=['titre', 'ville'], keep='first')
        df = df[(df['prix'].notna()) | (df['surface_m2'].notna())]
        if 'prix' in df.columns:
            df = df[(df['prix'].isna()) | ((df['prix'] >= 50000) & (df['prix'] <= 50000000))]
        if 'surface_m2' in df.columns:
            df = df[(df['surface_m2'].isna()) | ((df['surface_m2'] >= 10) & (df['surface_m2'] <= 10000))]
        df = df[df['ville'].str.len() > 0]
        df['type_bien'] = df['type_bien'].str.capitalize().fillna('Appartement')
        return df
        
    def combine(self):
        df_avito, df_mubawab = self.load_latest_files()
        df = pd.concat([self.normalize(df_avito), self.normalize(df_mubawab)], ignore_index=True)
        df = self.clean_data(df)
        
        # Features dérivées
        mask = (df['prix'].notna()) & (df['surface_m2'].notna()) & (df['surface_m2'] > 0)
        df.loc[mask, 'prix_m2'] = (df.loc[mask, 'prix'] / df.loc[mask, 'surface_m2']).round(2)
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join(DATA_PROCESSED, f"immobilier_maroc_{timestamp}.csv")
        os.makedirs(DATA_PROCESSED, exist_ok=True)
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        
        logger.info(f"Combine terminé. {len(df)} annonces sauvegardées dans {output_path}")
        return df, output_path

In [ ]:
combiner = DataCombiner()
df_combined, latest_csv_path = combiner.combine()
df_combined.head()

## 3. Chargement PostgreSQL (Schéma en Étoile)

In [ ]:
def get_dim_map(conn, table, key_cols, val_col):
    with conn.cursor() as cur:
        cur.execute(f"SELECT {','.join(key_cols)}, {val_col} FROM {table}")
        rows = cur.fetchall()
        if len(key_cols) == 1:
            return {row[0]: row[1] for row in rows}
        else:
            return {tuple(row[:-1]): row[-1] for row in rows}

def load_csv_to_sql(csv_path):
    df = pd.read_csv(csv_path)
    if df.empty:
        logger.warning("Le CSV est vide.")
        return

    # Nettoyage de base
    df["prix"] = pd.to_numeric(df["prix"].replace(r'[^\d.]', '', regex=True), errors="coerce")
    if "surface_m2" in df.columns:
        df["surface_m2"] = pd.to_numeric(df["surface_m2"], errors="coerce")
        df.loc[df["surface_m2"] > 50000, "surface_m2"] = None

    for col in ["parking", "ascenseur", "balcon", "piscine", "jardin"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    if 'quartier' not in df.columns: df['quartier'] = 'Inconnu'
    df['quartier'] = df['quartier'].fillna('Inconnu')
    df['ville'] = df['ville'].fillna('Inconnu')
    df['source'] = df['source'].fillna('Inconnu')
    df['type_bien'] = df['type_bien'].fillna('Appartement')

    conn = psycopg2.connect(**DB_CONFIG)
    try:
        with conn.cursor() as cur:
            # 1. DIMENSION SOURCE
            sources = df['source'].unique()
            execute_values(cur, "INSERT INTO dim_source (site_source) VALUES %s ON CONFLICT (site_source) DO NOTHING", [(x,) for x in sources])
            conn.commit()
            dim_source_map = get_dim_map(conn, 'dim_source', ['site_source'], 'id_source')

            # 2. DIMENSION TYPE DE BIEN
            types = df['type_bien'].unique()
            execute_values(cur, "INSERT INTO dim_type_bien (type_bien) VALUES %s ON CONFLICT (type_bien) DO NOTHING", [(x,) for x in types])
            conn.commit()
            dim_type_map = get_dim_map(conn, 'dim_type_bien', ['type_bien'], 'id_type')

            # 3. DIMENSION LOCALISATION
            locs = df[['ville', 'quartier']].drop_duplicates().values.tolist()
            execute_values(cur, "INSERT INTO dim_localisation (ville, quartier) VALUES %s ON CONFLICT (ville, quartier) DO NOTHING", locs)
            conn.commit()
            dim_loc_map = get_dim_map(conn, 'dim_localisation', ['ville', 'quartier'], 'id_loc')

            # Mapping des Foreign Keys
            df['id_source'] = df['source'].map(dim_source_map)
            df['id_type'] = df['type_bien'].map(dim_type_map)
            df['id_loc'] = df.apply(lambda row: dim_loc_map.get((row['ville'], row['quartier'])), axis=1)

            # 4. TABLE DE FAITS
            fact_cols = [
                'id_annonce', 'url', 'titre', 'prix', 'surface_m2', 
                'nb_chambres', 'nb_salles_bain', 'etage', 'parking', 
                'ascenseur', 'balcon', 'piscine', 'jardin', 'description', 
                'date_scraping', 'id_loc', 'id_type', 'id_source'
            ]
            final_cols = [c for c in fact_cols if c in df.columns]

            df_facts = df[final_cols].where(pd.notnull(df), None)
            records = df_facts.values.tolist()

            insert_query = f"""
                INSERT INTO fact_annonces ({','.join(final_cols)})
                VALUES %s
                ON CONFLICT (url) DO UPDATE SET
                    prix = EXCLUDED.prix,
                    surface_m2 = EXCLUDED.surface_m2,
                    updated_at = CURRENT_TIMESTAMP
            """
            execute_values(cur, insert_query, records)
            conn.commit()
            
            logger.info(f"\u2705 {len(records)} lignes injectées dans la table des faits (fact_annonces)")

    finally:
        conn.close()


In [ ]:
# Lancement de l'insertion SQL
load_csv_to_sql(latest_csv_path)